### Library Imports and ADK Model Initialization
This cell imports all necessary libraries for the Google ADK (Agent Development Kit) framework and defines an asynchronous function, `initialize_adk_model`, to set up the environment. The function loads API keys from environment variables, essential for authenticating with Google services and GitHub, ensuring secure access for the ADK agents.

In [1]:
import os
import re
import shutil
import tempfile
import asyncio
import time
from IPython.display import display, Markdown
from google.adk.models.google_llm import Gemini
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent
from google.adk.runners import Runner
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool, ToolContext
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import load_memory, preload_memory
from google.adk.code_executors import BuiltInCodeExecutor
from google.adk.apps.app import App, ResumabilityConfig
from google.genai import types
from dotenv import load_dotenv


async def initialize_adk_model():
    """Initializes the Google ADK LLM agent with Gemini model.
    Loads API keys from environment variables and sets them for the session.

    Returns:
        A dictionary containing the loaded API key and GitHub token.

    Raises:
        ValueError: If GOOGLE_API_KEY or GITHUB_TOKEN environment variables are not set.
    """
    load_dotenv()  # Load environment variables from .env file
    api_key = os.getenv("GOOGLE_API_KEY")
    github_token = os.getenv("GITHUB_TOKEN")
    os.environ["GOOGLE_API_KEY"] = api_key
    os.environ["GITHUB_TOKEN"] = github_token
    if not api_key:
        raise ValueError("GOOGLE_API_KEY environment variable not set.")
    if not github_token:
        raise ValueError("GITHUB_TOKEN environment variable not set.")
    
    return {"api_key": api_key, "github_token": github_token}  


### HTTP Retry Configuration
This cell defines `retry_config`, an instance of `types.HttpRetryOptions`. This configuration specifies the retry policy for HTTP requests made by the agents, including the maximum number of attempts, the exponential backoff base, initial delay, and a list of HTTP status codes that should trigger a retry. This enhances the robustness of API calls by handling transient network issues or service unavailability gracefully.

In [2]:
# Configure HTTP retry options for API calls to ensure robustness
retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier for exponential backoff
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)


### File System and GitHub Interaction Utilities
This cell provides a suite of utility functions designed to interact with the file system and GitHub repositories. 

- **`read_file`**: Reads the content of a specified file.
- **`read_directory_files`**: Recursively reads all files within a given directory and its subdirectories, excluding files from `.git` folders.
- **`read_github_repository`**: Clones a public GitHub repository into a temporary directory and then reads the contents of all its files. It handles URL validation, cloning, and error management.
- **`cleanup_temp_directory`**: Safely removes a specified temporary directory and all its contents.
- **`get_linting_score`**: Executes Pylint on a given Python file or directory to assess code style and quality, returning a percentage-based linting score. It handles cases where Pylint might not be installed or if the path is invalid.

In [3]:
def read_file(file_path: str) -> str:
    """
    Reads the content of a file.

    Args:
        file_path: The path to the file.

    Returns:
        The content of the file as a string.
    """
    if not os.path.exists(file_path):
        return f"Error: File not found at {file_path}"
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file: {e}"


def read_directory_files(directory_path: str) -> dict[str, str]:
    """
    Reads the content of all files in a directory and its subdirectories.

    Args:
        directory_path: The path to the directory.

    Returns:
        A dictionary where keys are file paths and values are file contents.
    """
    if not os.path.isdir(directory_path):
        return {"error": f"Error: Directory not found at {directory_path}"}
    
    file_contents = {}
    for root, _, files in os.walk(directory_path):
        for file in files:
            file_path = os.path.join(root, file)
            # Skip .git directory files
            if '.git' in file_path.split(os.sep):
                continue
            file_contents[file_path] = read_file(file_path)
            
    return file_contents


async def read_github_repository(repo_url: str) -> dict:
    """
    Clones a public GitHub repository and reads the content of all its files.
    Returns the temporary directory path where the repository was cloned, along with file contents.

    Args:
        repo_url: The full URL of the GitHub repository to clone.

    Returns:
        A dictionary with keys 'file_contents' (dictionary of file paths and contents) and 'temp_dir' (path to the temporary directory),
        or an error message.
    """
    print(f"Debug: read_github_repository received URL: {repo_url}") # Debug print
    # Validate the GitHub URL - more flexible regex
    if not re.match(r"https://github\.com/([^/]+)/([^/]+)", repo_url):
        return {"error": "Invalid GitHub repository URL provided."}

    temp_dir = tempfile.mkdtemp()
    try:
        # Construct the git clone command
        command = f"git clone --depth 1 {repo_url} ."
        
        # Execute the command in the temporary directory
        process = await asyncio.create_subprocess_shell(
            command,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
            cwd=temp_dir
        )
        
        stdout, stderr = await process.communicate()

        if process.returncode != 0:
            # Clean up on clone failure
            if os.path.exists(temp_dir):
                shutil.rmtree(temp_dir)
            return {"error": f"Failed to clone repository. Error: {stderr.decode()}"}

        # Read the files from the cloned repository
        file_contents = read_directory_files(temp_dir)
        return {"file_contents": file_contents, "temp_dir": temp_dir}

    except Exception as e:
        # Clean up on unexpected error
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
        return {"error": f"An unexpected error occurred: {e}"}
            

def cleanup_temp_directory(directory_path: str) -> dict:
    """
    Removes a temporary directory and its contents.

    Args:
        directory_path: The path to the temporary directory to remove.

    Returns:
        A dictionary indicating success or an error message.
    """
    if not os.path.isdir(directory_path):
        return {"error": f"Error: Directory not found at {directory_path}"}
    try:
        shutil.rmtree(directory_path)
        return {"success": f"Successfully removed directory: {directory_path}"}
    except Exception as e:
        return {"error": f"Error removing directory {directory_path}: {e}"}


async def get_linting_score(path: str) -> dict:
    """
    Runs pylint on a given Python file or directory and returns its linting score.

    Args:
        path: The path to the Python file or directory to lint.

    Returns:
        A dictionary containing the linting score as a float and any errors,
        or an error message if the path is not found or pylint fails.
    """
    if not os.path.exists(path):
        return {"error": f"Error: Path not found at {path}"}
    
    # Check if it's a file and ensure it's a Python file
    if os.path.isfile(path) and not path.endswith('.py'):
        return {"error": "Error: Not a Python file."}

    try:
        # Run pylint as a subprocess with the absolute path
        absolute_path = os.path.abspath(path)
        # If it's a directory, pylint will try to lint modules inside it.
        # Adding --recursive=y is safer for modern pylint, but standard pylint <dir> often works too.
        command = f"pylint {str(absolute_path)}"
        
        process = await asyncio.create_subprocess_shell(
            command,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE
        )
        stdout, stderr = await process.communicate()

        pylint_output = stdout.decode()
        pylint_error = stderr.decode()

        # Pylint often returns non-zero exit codes for linting issues, so we don't strictly fail on returncode != 0
        if process.returncode != 0 and "No such file or directory" in pylint_error:
             return {"error": "Pylint is not installed or not found. Please install it using 'pip install pylint'."}
        
        # Extract the score using a regular expression
        match = re.search(r"Your code has been rated at ([-+]?\d*\.\d+|\d+)/10", pylint_output)
        if match:
            score_out_of_10 = float(match.group(1))
            percentage_score = score_out_of_10 * 10  # Convert to percentage
            return {"linting_score": percentage_score}
        else:
            return {
                "linting_score": 0.0, 
                "message": "Pylint ran but no score was found (possibly no python files or fatal errors).", 
                "pylint_output": pylint_output[:500] # Return partial output for debugging
            }

    except Exception as e:
        return {"error": f"An unexpected error occurred during linting: {e}"}

In [4]:
description_generator = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    name="description_generator",
    description="An agent specialized in generating analyses and description for the code.",
    instruction="""You are an expert software engineer with a deep understanding of artificial intelligence, specializing in 
                    generating comprehensive and clear code descriptions. Your task is to analyze the provided code (file, directory, or repository) 
                    and generate a detailed, easy-to-understand description.

                    Your description should:
                    1.  **Overview:** Start with a high-level summary of the code's purpose and overall architecture, mentioning the languages and libraries used, an any configuration or scripts modules.
                    2.  **Component Breakdown:** Identify key files, classes, and functions, explaining their individual roles and how they contribute to the overall system.
                    3.  **Functionality Summary:** Conclude with a concise, single-line summary of the code's primary functionality.

                    Ensure your response is structured in Markdown for readability. Do not include raw code content in your output.
                    """,
    tools=[ FunctionTool(read_file),
            FunctionTool(read_directory_files),
            FunctionTool(read_github_repository),
            FunctionTool(get_linting_score),
            FunctionTool(cleanup_temp_directory),
            ],
    output_key="description_report",
)

correctness_assessor = LlmAgent(
    model=Gemini(model="gemini-2.0-flash", retry_options=retry_config),
    name="correctness_assessor",
    description="An agent specialized in code correctness assessment.",
    instruction="""
        You are a highly experienced coding expert specializing in code correctness assessment. Your task is to meticulously evaluate the provided code or codebase based on the following criteria:

        1.  **Functionality:** Assess if the code robustly delivers its requirements, effectively handles errors, manages invalid inputs gracefully, and considers common and edge cases.
        2.  **Security:** Identify any potential security vulnerabilities, common exploits, or insecure practices.
        3.  **Resource Efficiency & Memory Optimization:** Evaluate the code's efficiency in terms of CPU usage, memory consumption, and overall resource management.
        4.  **Test Coverage:** If a repository is provided, determine the presence and quality of automated tests (unit, integration, etc.). Note if tests are absent or poorly implemented.

        For each of these four points, provide a concise analysis (2-3 sentences max) and a corresponding percentage score (0-100%). Finally, synthesize these into an overall code correctness score (0-100%). 
        Present your assessment in a clear, structured Markdown format, with a dedicated section for each point and the final overall score.
        """,
    tools=[ FunctionTool(read_file),
            FunctionTool(read_directory_files),
            FunctionTool(read_github_repository),
            FunctionTool(get_linting_score),
            FunctionTool(cleanup_temp_directory),
            ],
    output_key="correctness_assessment",
)


style_assessor = LlmAgent(
    model=Gemini(model="gemini-2.0-flash", retry_options=retry_config),
    name="style_assessor",
    description="An agent specialized in code style assessment.",
    instruction="""You are an expert code style assessor. Your task is to provide a structured code style assessment report for the provided code or repository.

            **For any code input (file or directory):**
            Evaluate the code based on:
                -   **Readability:** Descriptive naming, consistent formatting, comments.
                -   **Maintainability & Organization:** Structure, modularity, large functions, duplication.
                -   **Python Linting:** Use the score obtained from `get_linting_score`.
                -   **Repository Best Practices:** Presence of `README.md`, `.gitignore`, `requirements.txt`, etc.

            Provide a concise analysis (2-3 sentences) and a percentage score (0-100%) for each aspect. Conclude with an overall code style score (0-100%). Present your assessment in clear Markdown with dedicated sections 
            for each point and the final overall score. **Do not output raw file contents.**
            """,
    tools=[ FunctionTool(read_file),
            FunctionTool(read_directory_files),
            FunctionTool(read_github_repository),
            FunctionTool(get_linting_score),
            FunctionTool(cleanup_temp_directory),
            ],
    output_key="style_assessment",
)



In [5]:
parallel_assessment_agent = ParallelAgent(
    name="ParallelAssessmentAgent",
    sub_agents=[description_generator, correctness_assessor, style_assessor],
    description="Runs multiple assessment agents in parallel to gather information."
)

In [6]:
improvement_recommender = LlmAgent(
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    name="improvement_recommender",
    tools=[AgentTool(parallel_assessment_agent)],
    description="recommends valid improvements based on the report received",
    instruction="""You are an expert software engineer specializing in code improvement and refactoring. Your task is to analyze the detailed code description, correctness assessment, 
    and style assessment provided by other agents. Based on this comprehensive input, generate a list of actionable and specific recommendations for improving the code or codebase.
        
        Your recommendations should cover the following areas:
        
        1.  **Functionality & Correctness**: Suggest improvements related to identified bugs, error handling, edge cases, security vulnerabilities, resource efficiency, and test coverage.
        2.  **Code Style & Quality**: Recommend changes for readability (naming, formatting, comments), maintainability (structure, modularity, reducing duplication), and adherence to established linting rules.
        3.  **Repository Best Practices**: Propose additions or modifications to repository files like `README.md`, `.gitignore`, or `requirements.txt` if they are missing or inadequate.\n\n
        
        Each recommendation should be clear, concise, and provide enough detail for a developer to understand and implement it. Present your recommendations as a Markdown-formatted unordered list.""",
    output_key="improvement_recommendations",
)

In [7]:
sequential_pipeline_agent = SequentialAgent(
    name="CodeAssessmentPipeline",
    sub_agents=[parallel_assessment_agent, improvement_recommender],
    description="Coordinates parallel assessment and synthesizes the results."
)

In [8]:
report_generator = LlmAgent(
    model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
    name="report_generator",
    description="Compiles a detailed report based on code assessments and description.",
    tools=[AgentTool(sequential_pipeline_agent),],
       #     preload_memory,
       #     google_search,
       #     ],
    instruction="""You are a professional technical report generator. Your task is to receive the outputs from the `description_generator`, `correctness_assessor`, `style_assessor`, and `improvement_recommender` agents, and compile them into a single, comprehensive, and well-structured report.
    
    The report should be organized into the following Markdown-formatted sections:
    
    # Code Assessment Report
    # 
    # ## 1. Code Description
     - (description_report)
    # 
    # ## 2.Code Correctness Assessment
     - (correctness_assessment)
    #
    ## 3. Code Style Assessment
     - (style_assessment)
    #
    ## 4. Suggested Improvements
     - (improvement_recommendations)

    Ensure that you only synthesize and format the provided information. Do not generate new content, assessments, or scores. Maintain the original wording and  
    formatting of the sections received from other agents where appropriate.

    """,
    )

In [ ]:
import tempfile
import webbrowser
import os
from pathlib import Path

def display_report_in_browser(report_text: str, filename="code_assessment_report.html"):
    """
    Converts markdown report to HTML and opens it in the default browser.
    
    Args:
        report_text: The markdown content of the report
        filename: Name of the HTML file to create
    """
    # Enhanced HTML template with better styling
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Code Assessment Report</title>
        <style>
            body {{
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', sans-serif;
                line-height: 1.6;
                max-width: 900px;
                margin: 0 auto;
                padding: 20px;
                background: #f5f5f5;
            }}
            .container {{
                background: white;
                padding: 40px;
                border-radius: 8px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            }}
            h1 {{
                color: #2c3e50;
                border-bottom: 3px solid #3498db;
                padding-bottom: 10px;
            }}
            h2 {{
                color: #34495e;
                margin-top: 30px;
                border-bottom: 2px solid #ecf0f1;
                padding-bottom: 8px;
            }}
            h3 {{
                color: #7f8c8d;
            }}
            pre {{
                background-color: #f8f9fa;
                border-left: 4px solid #3498db;
                padding: 15px;
                border-radius: 4px;
                overflow-x: auto;
            }}
            code {{
                background-color: #f8f9fa;
                padding: 2px 6px;
                border-radius: 3px;
                font-family: 'Courier New', monospace;
            }}
            ul, ol {{
                padding-left: 25px;
            }}
            li {{
                margin-bottom: 8px;
            }}
            .score {{
                background: #e8f4f8;
                padding: 10px;
                border-radius: 5px;
                font-weight: bold;
            }}
            blockquote {{
                border-left: 4px solid #3498db;
                padding-left: 15px;
                color: #555;
                font-style: italic;
            }}
        </style>
        <!-- Include markdown-it for rendering -->
        <script src="https://cdn.jsdelivr.net/npm/markdown-it@13/dist/markdown-it.min.js"></script>
    </head>
    <body>
        <div class="container">
            <div id="content"></div>
        </div>
        <script>
            // Initialize markdown-it
            var md = window.markdownit();
            
            // Your markdown content
            var markdownText = `{report_text.replace('`', '\\`')}`;
            
            // Render markdown to HTML
            document.getElementById('content').innerHTML = md.render(markdownText);
        </script>
    </body>
    </html>
    """
    
    # Save to reports directory in the workspace
    reports_dir = Path("/home/samer/Desktop/studies/5-Day AI Agents Intensive kaggle/capstone_project/agents_intensive_dev/reports")
    reports_dir.mkdir(exist_ok=True)
    
    file_path = reports_dir / filename
    
    # with open(file_path, "w", encoding="utf-8") as f:
        # f.write(html_content)
    
    print(f"📄 Report saved to: {file_path}")
    
    # Open in browser
    webbrowser.open(f'file://{file_path}')
    print(f"🌐 Opening report in browser...")
    
    return str(file_path)

In [10]:

code_file_path = "/home/samer/Desktop/studies/5-Day AI Agents Intensive kaggle/capstone_project/agents_intensive_dev/src/tools/read_files.py"
code_dir_path = "<path>"
repo_http_path = "https://github.com/Samir-atra/Emotion_estimation_from_video_footage_with_LSTM_ML_algorithm"
USER_ID = "explorer"
session_id = "explorer_session"

code_broker_app = App(
    name="code_broker",
    root_agent=report_generator,
    resumability_config=ResumabilityConfig(is_resumable=False),
)

agent = await initialize_adk_model()
session_service = InMemorySessionService()
memory_service = InMemoryMemoryService()

try:
    session = await session_service.create_session(session_id=session_id, app_name="code_broker", user_id=USER_ID)
except:
    session = await session_service.get_session(session_id=session_id, app_name="code_broker", user_id=USER_ID)

# Create runner with the resumable app
coding_runner = Runner(
app=code_broker_app,  # Pass the app instead of the agent
session_service=session_service,
memory_service=memory_service,
)

query = [f"provide a report for the repository at:{repo_http_path}"]
user_message = types.Content(role="user", parts=[types.Part(text=query[0])])


# await coding_runner.run_debug(user_messages=f"provide a report for the repository at:{repo_http_path}")


async for event in coding_runner.run_async(
    user_id=USER_ID, session_id=session_id, new_message=user_message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text and text != "None":
            # print(f"Model: > {text}")
            display(Markdown("## 📊 Generated Report Preview"))
            display(Markdown(text))
            
            # Save and open in browser
            display_report_in_browser(text)



/tmp/ipykernel_106617/4287309618.py:10: UserWarning: [EXPERIMENTAL] ResumabilityConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  resumability_config=ResumabilityConfig(is_resumable=False),


Debug: read_github_repository received URL: https://github.com/Samir-atra/Emotion_estimation_from_video_footage_with_LSTM_ML_algorithm
Debug: read_github_repository received URL: https://github.com/Samir-atra/Emotion_estimation_from_video_footage_with_LSTM_ML_algorithm


## 📊 Generated Report Preview

# Code Assessment Report
# 
## 1. Code Description
 - (No description report available from the assessment pipeline.)
# 
## 2. Code Correctness Assessment
 - (No correctness assessment available from the assessment pipeline.)
#
## 3. Code Style Assessment
 - (No style assessment available from the assessment pipeline.)
#
## 4. Suggested Improvements
 - Here are the actionable and specific recommendations for improving the codebase:

### Functionality & Correctness

*   **Dynamic Array Sizing**: Replace hardcoded array dimensions in `src/model/evaluation.py`, `src/data/data_loading.py`, and `src/data/dataset_indexing.py` (e.g., `(1646, 52, 1)`, `(22515, 52, 1)`) with dynamic calculations based on actual data size (e.g., `len(data)` or `data.shape[0]`). This will prevent errors when dataset sizes change.
*   **Centralize GPU Configuration**: Consolidate the TensorFlow GPU configuration (e.g., `tf.config.experimental.set_synchronous_execution(True)` and `tf.config.set_logical_device_configuration`) into a single dedicated utility function in `src/utils/gpu_config.py`. Ensure this function is called once at the application's entry point to avoid redundant configurations and potential conflicts.
*   **Correct Normalization Application**: In `src/data/augmenting_and_normalizing.py`, ensure the `Normalization` layer is adapted only once to the training data (`norm.adapt(x_train)`). Then, apply this *same* fitted `norm` object to both the training and validation sets to ensure consistent scaling.
*   **Standardize CSV Writer Header**: Modify the `csv_writer` function in `src/utils/csv_writer.py` to consistently expect a list of strings for the `fields` (header) argument. Update all calls to `csv_writer` (e.g., in `src/mediapipe_tools/blendshapes_dataset.py` where "beedoo" is used, and in `src/utils/error_analysis.py`) to pass an actual list of column names.
*   **Remove Redundant Sleep Calls**: Eliminate `time.sleep(0)` from `src/mediapipe_tools/choosing_blendshapes.py`. If performance is an issue, investigate and optimize the underlying image processing and MediaPipe detection logic directly instead of introducing artificial delays.
*   **Clarify Function Parameters**:
    *   In `src/utils/prediction_and_latency.py` and `src/utils/error_analysis.py`, refactor `generate_prediction(x_test)` and `count_visualize_errors(x_test)` to either genuinely accept `x_test` as an input and use it, or rename the parameters to clearly indicate that data is loaded internally (e.g., `_`).
    *   Update the docstrings accordingly.
*   **Implement Confusion Matrix Plotting**: The function `plot_confusion_matrix` in `src/utils/confusion_matrix.py` currently calculates but does not plot the confusion matrix. Either implement the plotting functionality as implied by its name or rename the function to `calculate_confusion_matrix` for clarity.
*   **Enforce Script Execution Guards**: Wrap all top-level executable code within `if __name__ == "__main__":` blocks in `src/data/augmenting_and_normalizing.py`, `src/data/dataset_indexing.py`, and `src/utils/gpu_config.py`. This prevents unintended side effects when these files are imported as modules.
*   **Remove Unused Code**: Identify and remove unused variables (e.g., `full_set`, `indices` in `src/mediapipe_tools/blendshapes_dataset.py`) and unused imports (e.g., `matplotlib.pyplot as plt` in `src/data/data_processing.py`) to improve code cleanliness and reduce cognitive load. Re-evaluate the module-level call to `sets_cleaner` in `src/data/augmenting_and_normalizing.py`.
*   **Externalize Configuration Paths**: Centralize all absolute paths (e.g., model paths, dataset paths, PythonPath additions) that are currently hardcoded in files like `src/model/evaluation.py`, `src/script.sh`, and `src/data/data_cleaning.py` by defining them in the `.env` file and accessing them via `os.getenv()`. This significantly improves portability and security.
*   **Robust Error Handling**: Implement `try-except` blocks for file operations (e.g., `open()`, `tf.io.read_file()`) across the codebase to gracefully handle `FileNotFoundError`, `IOError`, and other potential exceptions, making the application more resilient.
*   **Add Comprehensive Tests**: Develop a comprehensive suite of unit tests for individual functions and integration tests for workflows (e.g., data pipeline, model training, evaluation). This is crucial given the current low test coverage score (20%).

### Code Style & Quality

*   **Improve Linting Adherence**: Address all issues reported by linters like Pylint (current score: 29.7/100). Focus on improving code readability, consistency, and adherence to Python best practices, including proper variable naming, line lengths, and function complexity.
*   **Refactor Overly Large Functions**: Break down large functions, suchs as `categories_and_unreadable_counter` and `balanced_dataset` in `src/data/data_processing.py`, and `count_visualize_errors` in `src/utils/error_analysis.py`, into smaller, more focused functions. Each function should ideally have a single, well-defined responsibility.
*   **Eliminate Code Duplication**: Identify and refactor duplicated code segments.
    *   **Image Processing**: Create a reusable utility function (e.g., `pixels_to_mediapipe_image`) that encapsulates the logic for converting pixel strings to MediaPipe image objects. This is currently repeated in `src/data/data_processing.py`, `src/data/data_cleaning.py`, `src/data/augmenting_and_normalizing.py`, `src/mediapipe_tools/blendshapes_dataset.py`, `src/mediapipe_tools/choosing_blendshapes.py`, and `src/utils/error_analysis.py`.
    *   **Data Loading**: Develop a generic data loading function or class that can read CSV files and perform initial preprocessing steps, reducing redundancy across `src/data/data_loading.py`, `src/model/evaluation.py`, and `src/utils/prediction_and_latency.py`.
*   **Apply Consistent Formatting**: Use an automated code formatter (e.g., Black, autopep8) to ensure consistent line spacing, indentation, and overall code style across all Python files. This will improve readability and maintainability.
*   **Enhance Docstrings**: Review and refine docstrings for clarity, completeness, and accuracy. Ensure that all parameters, return values, and potential side effects are clearly documented. Specifically, update the `csv_writer` and `generate_prediction` docstrings to reflect the expected input types and actual function behavior.


NameError: name 'display_report_in_browser' is not defined

In [ ]:
exploring_session = await session_service.get_session(
    app_name="code_broker", user_id=USER_ID, session_id=session_id
)

await memory_service.add_session_to_memory(exploring_session)

In [ ]:
search_response = await memory_service.search_memory(
    app_name="code_broker", user_id=USER_ID, query="What are previous reports about?"
)

print("🔍 Search Results:")
print(f"  Found {len(search_response.memories)} relevant memories")
print()

for memory in search_response.memories:
    if memory.content and memory.content.parts:
        text = memory.content.parts[0].text[:80]
        print(f"  [{memory.author}]: {text}...")

🔍 Search Results:
  Found 1 relevant memories

  [report_generator]: # Code Assessment Report
# 
# ## 1. Code Description
 - Unfortunately, a compreh...


Gtk-Message: 16:15:46.032: Failed to load module "xapp-gtk3-module"
Gtk-Message: 16:15:46.032: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
[99220, Main Thread] WARNING: GTK+ module /snap/firefox/7355/gnome-platform/usr/lib/gtk-2.0/modules/libcanberra-gtk-module.so cannot be loaded.
GTK+ 2.x symbols detected. Using GTK+ 2.x and GTK+ 3 in the same process is not supported.: 'glib warning', file /build/firefox/parts/firefox/build/toolkit/xre/nsSigHandlers.cpp:201

(firefox_firefox:99220): Gtk-WARNING **: 16:15:46.128: GTK+ module /snap/firefox/7355/gnome-platform/usr/lib/gtk-2.0/modules/libcanberra-gtk-module.so cannot be loaded.
GTK+ 2.x symbols detected. Using GTK+ 2.x and GTK+ 3 in the same process is not supported.
Gtk-Message: 16:15:46.128: Failed to load module "canberra-gtk-module"
[99220, Main Thread] WARNING: GTK+ module /snap/firefox/7355/gnome-platform/usr/lib/gtk-2.0/modules/libcanberra-gtk-module.so cannot be load